# Feature Engineering for Time-Series Forecasting
### Teaching Notebook 02 | Data Science Education Series
**Author:** Md Minhazur Rahman 
**MSc Data Science (Merit)** — University of Greenwich 
**Preprint:** [DOI: 10.5281/zenodo.19479285](https://doi.org/10.5281/zenodo.19479285) 
**Live Research Demo:** [Streamlit Dashboard](https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/) 

---

> **Prerequisites:** Complete [Notebook 01 — Introduction to Time-Series Forecasting](./01_intro_time_series_forecasting.ipynb) before starting this notebook.

> **Who is this for?**
> Students who have built their first forecasting model and want to understand why raw time-series data is not enough — and how to engineer features that give ML models the information they need to make accurate predictions.

---

## What You Will Learn

| Section | Topic | Key Concept |
|---|---|---|
| 1 | Why raw data is not enough | ML models cannot see patterns in raw sequences |
| 2 | Lag features | Giving the model a memory of the past |
| 3 | Rolling window features | Summarising recent history |
| 4 | Feature leakage | The most dangerous mistake in time-series ML |
| 5 | Improved model comparison | Does feature engineering actually help? |

---

## Section 1: Why Raw Time-Series Data Is Not Enough

In Notebook 01, we trained a Linear Regression model using only one feature: the **day number**.

```python
X = day_index  # just the day number: 1, 2, 3, ..., 100
y = sales      # what we want to predict
```

The model learned a straight line through the data. It captured the upward trend but **completely missed the weekly seasonality** — the regular ups and downs that repeat every 7 days.

### Why did it miss the pattern?

Because the model had no information about **what happened recently**. It could not answer:
- What were sales like yesterday?
- What was the average sales over the last week?
- Is today a typically high or low day?

**This is the fundamental problem in time-series ML:**

> A machine learning model cannot see a sequence. It sees only the features you give it as columns in a table. Your job as a data scientist is to extract the temporal patterns from the sequence and encode them as explicit features.

### The Solution: Feature Engineering

**Feature engineering** means creating new input columns that capture information the model cannot extract on its own.

For time-series, the two most important types of engineered features are:

1. **Lag features** — what was the value at a previous time step?
2. **Rolling window features** — what was the average/spread over a recent window?

Let us build both from scratch.

---

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Reproduce the same dataset from Notebook 01
np.random.seed(42)
n_days = 100
day_index = np.arange(1, n_days + 1)

trend    = 0.3 * day_index
seasonal = 10 * np.sin(2 * np.pi * day_index / 7)
noise    = np.random.normal(0, 4, size=n_days)
sales    = 40 + trend + seasonal + noise

df = pd.DataFrame({
    'day':   day_index,
    'sales': sales
})

print('Dataset recreated successfully.')
print(f'Shape: {df.shape}')
print(df.head(10).to_string(index=False))

## Section 2: Lag Features

### What is a lag feature?

A **lag feature** is the value of the target variable at a previous time step, used as an input to predict the current time step.

| Day | Sales | sales_lag_1 | sales_lag_7 |
|---|---|---|---|
| 1 | 52.3 | NaN | NaN |
| 2 | 48.1 | 52.3 | NaN |
| 8 | 55.6 | 49.2 | 52.3 |
| 9 | 61.0 | 55.6 | 48.1 |

- **sales_lag_1:** Yesterday's sales. Captures short-term momentum.
- **sales_lag_7:** Sales exactly one week ago. Captures weekly seasonality.

### Why does this work?

If sales were high yesterday, they are often high today too — there is **autocorrelation** in most real-world time-series. By giving the model yesterday's value as a feature, we are explicitly telling it: *use recent history to predict the future.*

The lag-7 feature is particularly powerful for weekly data because it tells the model: *last Monday's sales are a strong predictor of this Monday's sales.*

In [ ]:
# ── Create lag features ────────────────────────────────────────────────

# Lag 1: yesterday's sales
df['sales_lag_1'] = df['sales'].shift(1)

# Lag 7: sales exactly one week ago
df['sales_lag_7'] = df['sales'].shift(7)

print('Lag features created.')
print()
print('First 12 rows (notice NaN values at the top):')
print(df[['day', 'sales', 'sales_lag_1', 'sales_lag_7']]
      .head(12).to_string(index=False))
print()
print('Why NaN? Because day 1 has no "yesterday" and')
print('days 1-7 have no "one week ago".')
print('These rows will be dropped before training.')

In [ ]:
# Visualise the lag relationship
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: sales vs sales_lag_1
axes[0].scatter(df['sales_lag_1'], df['sales'],
                alpha=0.6, color='steelblue', edgecolors='white')
axes[0].set_xlabel('Yesterday\'s Sales (Lag 1)')
axes[0].set_ylabel('Today\'s Sales')
axes[0].set_title('Today vs Yesterday\n(Strong positive correlation expected)',
                   fontweight='bold')

# Plot 2: sales vs sales_lag_7
axes[1].scatter(df['sales_lag_7'], df['sales'],
                alpha=0.6, color='darkorange', edgecolors='white')
axes[1].set_xlabel('Sales 7 Days Ago (Lag 7)')
axes[1].set_ylabel('Today\'s Sales')
axes[1].set_title('Today vs Same Day Last Week\n(Captures weekly seasonality)',
                   fontweight='bold')

plt.tight_layout()
plt.show()

# Print correlation values
corr_lag1 = df['sales'].corr(df['sales_lag_1'])
corr_lag7 = df['sales'].corr(df['sales_lag_7'])
print(f'Correlation between today and yesterday:       {corr_lag1:.3f}')
print(f'Correlation between today and 7 days ago:      {corr_lag7:.3f}')
print()
print('A correlation above 0.7 is considered strong.')
print('Both lag features carry meaningful predictive information.')

## Section 3: Rolling Window Features

### What is a rolling window feature?

A **rolling window feature** summarises the recent history of the time-series over a fixed window of past observations.

The two most common rolling features are:
- **Rolling mean:** The average sales over the last N days
- **Rolling standard deviation:** The variability of sales over the last N days

| Day | Sales | rolling_mean_7 | rolling_std_7 |
|---|---|---|---|
| 7 | 55.2 | 49.8 | 4.2 |
| 8 | 61.0 | 51.3 | 4.7 |
| 9 | 48.5 | 52.1 | 5.1 |

- **rolling_mean_7** answers: *What has the typical sales level been over the past week?*
- **rolling_std_7** answers: *How volatile have sales been recently?*

### The Critical Rule: Always Shift After Rolling

> **You must shift rolling features by at least 1 after computing them.**

If you do not shift, the rolling mean for day N will include day N itself — meaning you are using today's actual value to predict today's value. This is **data leakage** and will be explained in detail in Section 4.

The correct approach:
```python
# WRONG — includes today's value in the window
df['rolling_mean_7'] = df['sales'].rolling(7).mean()

# CORRECT — window ends yesterday, not today
df['rolling_mean_7'] = df['sales'].shift(1).rolling(7).mean()
```

In [ ]:
# ── Create rolling window features ─────────────────────────────────────
# IMPORTANT: shift(1) before rolling to prevent leakage

# 7-day rolling mean (average of last 7 days, not including today)
df['rolling_mean_7'] = df['sales'].shift(1).rolling(window=7).mean()

# 7-day rolling standard deviation (volatility of last 7 days)
df['rolling_std_7']  = df['sales'].shift(1).rolling(window=7).std()

print('Rolling features created (with shift to prevent leakage).')
print()
print('Rows 8 to 15 (first rows with complete features):')
print(df[['day', 'sales', 'rolling_mean_7', 'rolling_std_7']]
      .iloc[7:15].to_string(index=False))

In [ ]:
# Visualise rolling mean vs actual sales
plt.figure(figsize=(13, 4))

plt.plot(df['day'], df['sales'],
         color='steelblue', linewidth=1.2,
         alpha=0.7, label='Actual Daily Sales')

plt.plot(df['day'], df['rolling_mean_7'],
         color='red', linewidth=2,
         label='7-Day Rolling Mean (lagged)')

plt.fill_between(
    df['day'],
    df['rolling_mean_7'] - df['rolling_std_7'],
    df['rolling_mean_7'] + df['rolling_std_7'],
    alpha=0.15, color='red',
    label='Rolling Mean +/- 1 Std Dev'
)

plt.title('Actual Sales vs 7-Day Rolling Mean',
           fontsize=13, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Units Sold')
plt.legend()
plt.tight_layout()
plt.show()

print('The red line smooths out daily noise and tracks the underlying trend.')
print('The shaded band shows how variable sales have been recently.')
print('Both are useful signals for a forecasting model.')

## Section 4: Feature Leakage — The Most Dangerous Mistake

### What is feature leakage?

**Feature leakage** occurs when your model is trained on information that would not be available at the time of making a real prediction.

In time-series forecasting, leakage almost always happens in one of two ways:

### Type 1: Rolling Without Shifting

```python
# This looks harmless but is WRONG
df['rolling_mean_7'] = df['sales'].rolling(7).mean()
```

The rolling mean for day 10 would include days 4, 5, 6, 7, 8, 9, **and 10**.
But on day 10, you do not yet know day 10's actual sales — that is what you are trying to predict.
You are accidentally giving the model the answer inside the question.

### Type 2: Random Train-Test Split

```python
# NEVER do this for time-series
from sklearn.model_selection import train_test_split
X_train, X_test = train_test_split(X, test_size=0.2, shuffle=True)
```

If you shuffle the data randomly, your training set will contain future observations.
The model learns from the future to predict the past — which is impossible in deployment.
Your evaluation metrics will look excellent but the model will fail completely in production.

### The Correct Approach: Temporal Split + Shifted Features

```python
# ALWAYS split by time
train = df[df['day'] <= 80]
test  = df[df['day'] >  80]

# ALWAYS shift before rolling
df['rolling_mean_7'] = df['sales'].shift(1).rolling(7).mean()
```

> **The rule is simple: at prediction time for day N, your model may only use information from days 1 through N-1.**

### Why This Matters in Practice

This is not a theoretical concern. Models trained with leakage have been deployed in production systems — in retail, finance, and healthcare — and failed catastrophically when they encountered real data where the future was genuinely unknown.

My MSc dissertation used **Rolling-Origin Cross-Validation (ROCV)** to prevent this entirely. ROCV is introduced in Notebook 03.

---

In [ ]:
# Demonstrate the danger of leakage with a concrete example
print('=== LEAKAGE DEMONSTRATION ===')
print()

# Leaky version: rolling without shift
df['leaky_rolling_mean'] = df['sales'].rolling(7).mean()

# Safe version: rolling with shift
df['safe_rolling_mean'] = df['sales'].shift(1).rolling(7).mean()

# Show the difference for a specific day
day_example = 20
row = df[df['day'] == day_example].iloc[0]

print(f'For Day {day_example}:')
print(f'  Actual sales (the answer):     {row["sales"]:.2f}')
print(f'  Leaky rolling mean:            {row["leaky_rolling_mean"]:.2f}')
print(f'  Safe rolling mean (shifted):   {row["safe_rolling_mean"]:.2f}')
print()
print('The leaky version includes today\'s actual sales in the mean.')
print('The safe version uses only days 13-19 (yesterday and before).')
print()
print('The difference looks small — but across thousands of predictions,')
print('it causes the model to appear far more accurate than it really is.')

## Section 5: Does Feature Engineering Actually Help?

We have engineered four new features:
- `sales_lag_1` — yesterday's sales
- `sales_lag_7` — sales one week ago
- `rolling_mean_7` — 7-day rolling average (lagged)
- `rolling_std_7` — 7-day rolling standard deviation (lagged)

Now let us answer the real question:

> **Does adding these features to our Linear Regression model actually improve its accuracy compared to Notebook 01?**

We will run a direct head-to-head comparison.

In [ ]:
# ── Prepare the feature-engineered dataset ─────────────────────────────

# Drop rows with NaN values created by lag and rolling operations
df_clean = df.dropna().copy()
print(f'Rows after dropping NaN: {len(df_clean)} (from {len(df)} original rows)')
print(f'First usable day: {df_clean["day"].min()}')
print()

# Define feature sets
features_baseline = ['day']
features_engineered = ['day', 'sales_lag_1', 'sales_lag_7',
                        'rolling_mean_7', 'rolling_std_7']

# Time-based train-test split (never shuffle)
split_point = 80
train = df_clean[df_clean['day'] <= split_point]
test  = df_clean[df_clean['day'] >  split_point]

print(f'Training rows: {len(train)}')
print(f'Test rows:     {len(test)}')

In [ ]:
# ── Train and evaluate both models ─────────────────────────────────────

def evaluate_model(train, test, features, model_name):
    model = LinearRegression()
    model.fit(train[features], train['sales'])
    predictions = model.predict(test[features])

    mae  = mean_absolute_error(test['sales'], predictions)
    rmse = np.sqrt(mean_squared_error(test['sales'], predictions))
    mape = np.mean(np.abs(
        (test['sales'] - predictions) / test['sales']
    )) * 100

    print(f'── {model_name} ──')
    print(f'  Features used: {features}')
    print(f'  MAE:   {mae:.2f} units')
    print(f'  RMSE:  {rmse:.2f} units')
    print(f'  MAPE:  {mape:.2f}%')
    print()
    return predictions, mae, mape

print('=== MODEL COMPARISON ===')
print()
pred_baseline, mae_baseline, mape_baseline = evaluate_model(
    train, test, features_baseline, 'Baseline (day only)')

pred_engineered, mae_engineered, mape_engineered = evaluate_model(
    train, test, features_engineered, 'Feature-Engineered Model')

improvement_mae  = ((mae_baseline - mae_engineered) / mae_baseline) * 100
improvement_mape = ((mape_baseline - mape_engineered) / mape_baseline) * 100

print(f'=== IMPROVEMENT FROM FEATURE ENGINEERING ===')
print(f'  MAE  reduced by: {improvement_mae:.1f}%')
print(f'  MAPE reduced by: {improvement_mape:.1f}%')

In [ ]:
# ── Visualise the comparison ───────────────────────────────────────────
plt.figure(figsize=(13, 5))

plt.plot(train['day'], train['sales'],
         color='steelblue', linewidth=1.2,
         label='Training Data')

plt.plot(test['day'], test['sales'],
         color='black', linewidth=1.8,
         label='Actual Sales (Test)')

plt.plot(test['day'], pred_baseline,
         color='red', linewidth=1.8,
         linestyle='--', alpha=0.8,
         label=f'Baseline (day only) — MAPE: {mape_baseline:.1f}%')

plt.plot(test['day'], pred_engineered,
         color='green', linewidth=2,
         linestyle='--',
         label=f'Feature-Engineered — MAPE: {mape_engineered:.1f}%')

plt.axvline(x=split_point, color='grey',
            linestyle=':', alpha=0.7,
            label='Train / Test Split')

plt.title('Baseline vs Feature-Engineered Model — Test Period',
           fontsize=13, fontweight='bold')
plt.xlabel('Day')
plt.ylabel('Units Sold')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Results and Interpretation

Look at the numbers carefully.

**The feature-engineered model uses the same algorithm** — Linear Regression — as the baseline. The only difference is what information we gave it.

This demonstrates the most important lesson in applied machine learning:

> **The quality of your features matters more than the complexity of your algorithm.**

A simple model with good features will almost always outperform a complex model with poor features.

---

## Discussion Questions for Students

1. We used lag-1 and lag-7. Can you think of other lag values that might be useful for retail data — for example, lag-14 or lag-30? What would they capture?

2. We computed a 7-day rolling mean. What would a 3-day rolling mean capture differently? When might a shorter window be more useful?

3. Why is feature leakage particularly dangerous in a business forecasting system? What real-world consequences could result from deploying a leaky model?

4. We showed that feature engineering improved a Linear Regression model. What do you predict will happen when we apply these same features to a Random Forest or XGBoost model in the next notebook?

---

## What Comes Next

In **Notebook 03**, we will:
- Replace Linear Regression with tree-based models (Random Forest and XGBoost)
- Apply the same engineered features to see how much more powerful they become
- Introduce Rolling-Origin Cross-Validation — the gold standard evaluation method for time-series
- Connect the complete pipeline to the live Streamlit dashboard

**Continue to:**
[Notebook 03 — Tree-Based Models and Rolling-Origin Cross-Validation](./03_tree_models_and_rocv.ipynb)

---

**Related Research:**
- Full pipeline: [github.com/minhazda/synthetic-retail-forecasting](https://github.com/minhazda/synthetic-retail-forecasting)
- Live demo: [retail-forecasting.streamlit.app](https://retail-forecasting-hvdzvesi4u9l6fs5tvdoyi.streamlit.app/)
- Published preprint: [DOI: 10.5281/zenodo.19479285](https://doi.org/10.5281/zenodo.19479285)

---
*Teaching materials for undergraduate data science education in Bangladesh.*
*© Md Minhazur Rahman, 2026. Open for educational use.*